# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pallavibt/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [18]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token saved successfully!" if HF_TOKEN else "Token not found.")

Token saved successfully!


In [19]:
%pip install -q huggingface_hub datasets duckdb pandas

In [20]:
from huggingface_hub import login

login(token=HF_TOKEN)
print("Connected to Hugging Face successfully.")

Connected to Hugging Face successfully.


In [21]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

print("Number of files:", len(files))
for f in files:
    print(f)

Number of files: 24
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_da

In [22]:
from huggingface_hub import hf_hub_download
import pandas as pd

repo = "FlyRank/internship-warehouse"

files_to_inspect = [
    "dim_clients.parquet",
    "dim_content.parquet",
    "fact_content_daily_performance/month=2026-03/data_0.parquet",
    "fact_content_query_90d.parquet",
]

for file_path in files_to_inspect:
    local_path = hf_hub_download(
        repo_id=repo,
        repo_type="dataset",
        filename=file_path,
        token=HF_TOKEN
    )

    df = pd.read_parquet(local_path)

    print("\n" + "=" * 60)
    print("FILE:", file_path)
    print("SHAPE:", df.shape)
    print("COLUMNS:", df.columns.tolist())
    print(df.head(3).to_string(index=False))


FILE: dim_clients.parquet
SHAPE: (104, 9)
COLUMNS: ['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']
         client_hash_id is_active has_gsc_access has_ga4_access                access_profile client_created_date client_updated_date gsc_data_start ga4_data_start
client_04660893ae39614a      True           True           True                   gsc_and_ga4          2026-04-15          2026-06-27           None     2026-05-22
client_05475c07ed21a83a      True          False          False no_search_or_analytics_access          2026-04-01          2026-06-27           None           None
client_06d356715a8ff3b6      True           True           True                   gsc_and_ga4          2026-03-23          2026-07-05     2026-04-10     2026-04-06

FILE: dim_content.parquet
SHAPE: (519606, 26)
COLUMNS: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_i

In [23]:
from huggingface_hub import hf_hub_download
import pandas as pd

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN
)

march = pd.read_parquet(march_path)

print("Rows:", len(march))
print("Columns:", len(march.columns))
print("All columns:")
for col in march.columns:
    print("-", col)

print("\nDate span:", march["report_date"].min(), "to", march["report_date"].max())
print("Unique report dates:", march["report_date"].nunique())

Rows: 9841378
Columns: 30
All columns:
- report_date
- client_hash_id
- content_hash_id
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ga4_users
- ga4_engaged_sessions
- ga4_total_engagement_sec
- sessions_organic
- sessions_direct
- sessions_referral
- sessions_social
- sessions_paid
- sessions_ai
- ai_chatgpt
- ai_perplexity
- ai_gemini
- ai_copilot
- ai_claude
- ai_meta
- ai_other
- scroll_events

Date span: 2026-03-01 to 2026-03-31
Unique report dates: 31


In [24]:
%pip install -q duckdb

In [25]:
import duckdb

con = duckdb.connect()

con.register("march", march)

# Query 1: Verify the grain
q1 = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id))
        AS unique_client_content_dates
FROM march
"""
print("QUERY 1 — GRAIN")
print(con.execute(q1).df().to_string(index=False))

# Query 2: Verify row count and date span
q2 = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(DISTINCT report_date) AS unique_dates
FROM march
"""
print("\nQUERY 2 — ROW COUNT AND DATE SPAN")
print(con.execute(q2).df().to_string(index=False))

# Query 3: Verify availability using IS TRUE
q3 = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)
        AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)
        AS ga4_available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
    ) AS both_available_rows
FROM march
"""
print("\nQUERY 3 — DATA AVAILABILITY")
print(con.execute(q3).df().to_string(index=False))

QUERY 1 — GRAIN


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  unique_client_content_dates
    9841378                      9841378

QUERY 2 — ROW COUNT AND DATE SPAN


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 row_count first_date  last_date  unique_dates
   9841378 2026-03-01 2026-03-31            31

QUERY 3 — DATA AVAILABILITY
 total_rows  gsc_available_rows  ga4_available_rows  both_available_rows
    9841378             3611061              413966               364347


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**1. Unit of analysis**

One row represents one client-content pair for one reporting date. I verified that all 984,138 rows have unique combinations of `report_date`, `client_hash_id`, and `content_hash_id`.

**2. Table(s) to use**

The main table is `fact_content_daily_performance`, using its March 2026 partition. The dimension tables `dim_content` and `dim_clients` can provide additional content or client context if needed.

**3. Time window**

March 1–31, 2026. This mid-panel month contains 984,138 rows across 31 reporting dates. June 2026 will remain the sealed final test month.

**4. What I want to rank**

I want to rank content pages for editor review based on search performance and traffic signals. The ranking target will be a proxy for content opportunity, not a confirmed historical editorial label.

**5. One thing deliberately excluded**

I will exclude future outcome information and label-derived fields from the model features to prevent data leakage.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features**

- `gsc_impressions`: measures how often a content page appears in Google Search.
- `gsc_clicks`: measures clicks received from Google Search.
- `gsc_avg_position`: indicates the page's average search position.
- `ga4_sessions`: measures website sessions for the content page.
- `ga4_engaged_sessions`: measures sessions that met the analytics engagement criteria.

**Label / proxy**

The goal is to rank pages for editorial review. There is no confirmed historical editorial-review label in the inspected table, so the ranking will use a clearly stated content-opportunity proxy rather than claiming to predict a verified outcome.

**Context**

The data is grouped by client and content page and reported daily. March 2026 is the analysis window. Data availability differs across rows, so GSC and GA4 availability flags must be considered.

**Excluded**

I will exclude future outcomes, label-derived columns, identifiers such as `client_hash_id` and `content_hash_id` from model inputs, and any fields that would not be available at the time the ranking is made. I will also avoid treating unavailable source values as genuine zeros.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Five candidate features and availability at decision time

1. **`gsc_impressions`** — Knowable at the decision moment because it records search impressions from the reporting data available up to the ranking date. Confirm the reporting delay before deployment.
2. **`gsc_clicks`** — Knowable at the decision moment because it records search clicks up to the ranking date, provided the reporting data has arrived.
3. **`gsc_avg_position`** — Knowable at the decision moment because it summarizes search position for the reporting period, provided GSC data is available. Missing values must remain distinguishable from genuine zero values.
4. **`ga4_sessions`** — Knowable at the decision moment because it records sessions in the analytics reporting period, provided GA4 data is available.
5. **`ga4_engaged_sessions`** — Knowable at the decision moment because it records engaged sessions in the reporting period, provided GA4 data is available.

**Availability caveat:** These fields are candidates for a ranking made after their reporting data has arrived. A production system must account for ingestion and reporting delays so it does not use information that was unavailable at the actual decision time.

In [28]:
# Create a feature frame from the March 2026 data.
feature_cols = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "gsc_data_available",
    "ga4_data_available",
]

feature_frame = march[feature_cols].copy()

print("Feature frame shape:", feature_frame.shape)
print("\nFirst five rows:")
display(feature_frame.head())

print("\nMissing values in candidate features:")
print(feature_frame[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_engaged_sessions",
    ]
].isna().sum())


Feature frame shape: (9841378, 10)

First five rows:


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,gsc_data_available,ga4_data_available
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,NaN,NaN,True,None
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,NaN,NaN,True,None
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,NaN,NaN,True,None
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,NaN,NaN,True,None
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,NaN,NaN,True,None



Missing values in candidate features:
gsc_impressions               0
gsc_clicks                    0
gsc_avg_position        6230317
ga4_sessions            3018741
ga4_engaged_sessions    3018741
dtype: int64


In [29]:
print("Available March columns:")
print(march.columns.tolist())

print("\nPotential outcome and label fields:")
keywords = ["label", "target", "outcome", "future", "next", "conversion", "purchase"]

matches = [
    col for col in march.columns
    if any(word in col.lower() for word in keywords)
]

print(matches if matches else "No obvious label/outcome field found.")

Available March columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']

Potential outcome and label fields:
No obvious label/outcome field found.


In [30]:
from huggingface_hub import hf_hub_download
import pandas as pd

query_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_query_90d.parquet",
    token=HF_TOKEN
)

queries = pd.read_parquet(query_path)

print("Shape:", queries.shape)
print("\nAll columns:")
for col in queries.columns:
    print("-", col)

print("\nFirst 3 rows:")
display(queries.head(3))

Shape: (2414248, 21)

All columns:
- client_hash_id
- content_hash_id
- query_hash_id
- query_char_count
- query_token_count
- window_start
- window_end
- impressions_90d
- clicks_90d
- impressions_last30
- clicks_last30
- impressions_prev30
- clicks_prev30
- avg_position_90d
- avg_position_last30
- avg_position_prev30
- content_total_impressions_90d
- content_visible_query_count
- rare_query_count
- rare_impressions_share
- anonymized_impressions_share

First 3 rows:


,client_hash_id,content_hash_id,query_hash_id,query_char_count,query_token_count,window_start,window_end,impressions_90d,clicks_90d,impressions_last30,...,impressions_prev30,clicks_prev30,avg_position_90d,avg_position_last30,avg_position_prev30,content_total_impressions_90d,content_visible_query_count,rare_query_count,rare_impressions_share,anonymized_impressions_share
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_58b1b001f839d699,17,3,2026-04-02,2026-06-30,11,0,0,...,11,0,10.818182,NaN,10.818182,1466,14,32,0.043656,0.725102
1,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_922b8eca2a24cd34,34,7,2026-04-02,2026-06-30,13,0,0,...,1,0,1.769231,NaN,11.000000,1466,14,32,0.043656,0.725102
2,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_9f0c36a6ae2a6a99,16,2,2026-04-02,2026-06-30,16,0,11,...,5,0,23.562500,24.272727,22.000000,1466,14,32,0.043656,0.725102


In [31]:
from huggingface_hub import hf_hub_download
import pandas as pd

clients_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_clients.parquet",
    token=HF_TOKEN
)

dim_clients = pd.read_parquet(clients_path)
print("Client history date columns:")
print(
    dim_clients[
        ["client_hash_id", "gsc_data_start", "ga4_data_start"]
    ].head()
)

Client history date columns:
            client_hash_id gsc_data_start ga4_data_start
0  client_04660893ae39614a           None     2026-05-22
1  client_05475c07ed21a83a           None           None
2  client_06d356715a8ff3b6     2026-04-10     2026-04-06
3  client_0797ff3a1fc9a6a5     2025-11-05           None
4  client_08a6a72ff48e62c0     2025-09-24           None


In [32]:
print("Query table date windows:")
print(
    queries[
        [
            "window_start",
            "window_end",
            "impressions_90d",
            "impressions_prev30",
            "impressions_last30",
            "clicks_prev30",
            "clicks_last30",
        ]
    ].head(10).to_string(index=False)
)

Query table date windows:
window_start window_end  impressions_90d  impressions_prev30  impressions_last30  clicks_prev30  clicks_last30
  2026-04-02 2026-06-30               11                  11                   0              0              0
  2026-04-02 2026-06-30               13                   1                   0              0              0
  2026-04-02 2026-06-30               16                   5                  11              0              0
  2026-04-02 2026-06-30               55                   1                   1              0              0
  2026-04-02 2026-06-30               14                   0                   0              0              0
  2026-04-02 2026-06-30               40                   3                   3              0              0
  2026-04-02 2026-06-30               16                   6                   5              0              0
  2026-04-02 2026-06-30               14                   2                   1      

In [33]:
import numpy as np

# Compare the 90-day totals with the two 30-day windows.
checks = {
    "impressions_90d": (
        "impressions_90d",
        "impressions_prev30",
        "impressions_last30",
    ),
    "clicks_90d": (
        "clicks_90d",
        "clicks_prev30",
        "clicks_last30",
    ),
}

for name, cols in checks.items():
    total, prev, last = cols
    comparable = queries[list(cols)].dropna()

    matches = (
        comparable[total]
        == comparable[prev] + comparable[last]
    )

    print(
        name,
        "| comparable rows:", len(comparable),
        "| exact matches:", int(matches.sum()),
        "| match rate:", round(matches.mean(), 4),
    )

impressions_90d | comparable rows: 2414248 | exact matches: 455025 | match rate: 0.1885
clicks_90d | comparable rows: 2414248 | exact matches: 2328345 | match rate: 0.9644


In [34]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Define a simple binary proxy from a real warehouse outcome field.
leak_df = queries[["clicks_prev30", "clicks_last30"]].dropna().copy()
leak_df["label"] = (leak_df["clicks_last30"] > 0).astype(int)

X = leak_df[["clicks_prev30"]]
y = leak_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Honest baseline: uses only the previous-window clicks.
honest_model = DecisionTreeClassifier(max_depth=2, random_state=42)
honest_model.fit(X_train, y_train)
honest_score = accuracy_score(y_test, honest_model.predict(X_test))

# Deliberately leaked version: directly includes the label as a feature.
X_leaked = leak_df[["clicks_prev30"]].copy()
X_leaked["LEAK_label_copy"] = leak_df["label"]

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leaked, y, test_size=0.25, random_state=42, stratify=y
)

leaked_model = DecisionTreeClassifier(random_state=42)
leaked_model.fit(Xl_train, yl_train)
leaked_score = accuracy_score(yl_test, leaked_model.predict(Xl_test))

print("Honest baseline accuracy:", round(honest_score, 4))
print("Deliberately leaked accuracy:", round(leaked_score, 4))

# Remove the leaked column; keep only the honest feature frame.
X_final = X_leaked.drop(columns=["LEAK_label_copy"])
print("Leaked column removed:", "LEAK_label_copy" not in X_final.columns)

Honest baseline accuracy: 0.9664
Deliberately leaked accuracy: 1.0
Leaked column removed: True


### Deliberate leakage experiment

**Proxy label:** Whether a query-level record received any clicks in the last 30-day window, defined as `clicks_last30 > 0`.

**Honest baseline:** Accuracy was 0.9664 (96.64%) using `clicks_prev30` as the only feature.

**Deliberately leaked model:** Accuracy increased to 1.0000 (100%) after adding `LEAK_label_copy`, a direct copy of the target label.

**Leakage lesson:** The perfect score is artificial because the feature reveals the answer. It must never be used as a model input.

**Cleanup:** The leaked column was removed, and the final feature frame retains only `clicks_prev30`.

**Limitation:** This experiment uses query-level records, not the page-day grain used for the main March data contract. The click proxy is not a verified editorial-review label, and the baseline result does not establish editorial-ranking quality.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limitations

1. **Uneven data availability:** GSC and GA4 data are not available for every row. In March 2026, 361,106 rows had GSC data available and 413,966 had GA4 data available. Missing values must not automatically be treated as zero.

2. **Daily reporting and delays:** The table records daily performance, but the data may arrive after the reporting date. A real ranking system must respect the information actually available at decision time.

3. **No confirmed editorial-review label:** The inspected tables do not expose an obvious verified label for whether a page needed a refresh or was successfully improved. The ranking target therefore remains a proxy until a suitable outcome is identified.

4. **Query-level versus page-level grain:** The query table contains query-level records and 90-day windows. These records must not be treated as interchangeable with daily client-content-page records.

5. **Limited time window:** This analysis uses March 2026. Results from one month may not represent other seasons or future months.

6. **Client history limitation:** Clients have different GSC and GA4 history start dates, and some have no recorded start date. Therefore, March 2026 does not guarantee complete historical coverage for every client. GA4 metrics must be interpreted using `ga4_data_available`, and GSC metrics using `gsc_data_available`, rather than treating unavailable data as zero.

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.